# Human Pose Estimation Using YOLO11

This notebook detects the 17 COCO human pose keypoints in an uploaded video and renders only a white skeleton on a black background.

**Model:** `yolo11n-pose.pt`  
**Libraries:** Ultralytics, OpenCV, NumPy, FFmpeg

In [ ]:
!pip install -q ultralytics opencv-python-headless

In [ ]:
import cv2
import numpy as np
import subprocess
from ultralytics import YOLO
from google.colab import files
from IPython.display import display, Video

# Load the YOLO11 pose-estimation model
model = YOLO("yolo11n-pose.pt")

# Upload your input video (for example: nlpvideo(1).mp4)
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload a video file.")
input_video = next(iter(uploaded))
print("Processing:", input_video)

In [ ]:
cap = cv2.VideoCapture(input_video)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if not cap.isOpened() or fps <= 0 or width <= 0 or height <= 0:
    cap.release()
    raise ValueError("Could not read the uploaded video. Check that it is a valid video file.")

output_video = "skeleton.mp4"
out = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# COCO-17 keypoint connections
connections = [
    (0, 1), (0, 2), (1, 3), (2, 4), (5, 6),
    (5, 7), (7, 9), (6, 8), (8, 10), (5, 11),
    (6, 12), (11, 12), (11, 13), (13, 15),
    (12, 14), (14, 16)
]

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    result = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.35,
        verbose=False
    )[0]

    # A black canvas: the original video frame is not copied into the output.
    skeleton = np.zeros_like(frame)

    # This version draws the first detected person in each frame.
    if result.keypoints is not None and len(result.keypoints.data) > 0:
        points = result.keypoints.data[0].cpu().numpy()

        for a, b in connections:
            if points[a, 2] > 0.25 and points[b, 2] > 0.25:
                p1 = tuple(points[a, :2].astype(int))
                p2 = tuple(points[b, :2].astype(int))
                cv2.line(skeleton, p1, p2, (255, 255, 255), 4)

        for x, y, confidence in points:
            if confidence > 0.25:
                cv2.circle(
                    skeleton, (int(x), int(y)), 6,
                    (255, 255, 255), -1
                )

    out.write(skeleton)

cap.release()
out.release()
print("Pose-only video created:", output_video)

In [ ]:
# Convert to a widely compatible MP4 format
subprocess.run([
    "ffmpeg", "-y", "-i", "skeleton.mp4",
    "-vcodec", "libx264", "-pix_fmt", "yuv420p",
    "skeleton_final.mp4"
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Preview and download the result
display(Video("skeleton_final.mp4", width=700))
files.download("skeleton_final.mp4")